# Проект III: Интерактивная карта на тему доступности зелени

- Цель: создать карту с границами района, участками землепользования, буферами доступностями и зданиями.
- Буферы доступности будут строится от "зеленных" типов землепользования
- Здания которые будут входить в буфер доступности будут отмечаться зеленным(которые не будут входить красным)
- Результат: Карта на которой показано какие здания в городе входят в буфер доступности зеленных насаждений


## 0. Импорт библиотек и подготовка данных

### 0.1 Импорт библиотек


In [ ]:
import geopandas as gpd
import folium
import osmnx as ox
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx


### 0.2 Подготовка данных

Мы используем данные из OSM


#### 0.2.1. Границы района


Читаем файл с границами района.


In [ ]:

place_name = "Ленинский район, Екатеринбург"

area = ox.geocode_to_gdf(place_name)


Смотрим структуру данных.


In [ ]:

area.head()

Отображаем слой на карте.


In [ ]:
area.explore(tiles="cartodbpositron")

#### 0.2.2. Землепользование


- Переводим район в метрическую систему координат, добавлям буфер 300 метров, и возвращаем обратно в географическую 
- Делаем мы это для того, потому что соседнии с нашей территорией участки землепользования тоже оказывают на нашу территорию влияние.

In [ ]:

district_buffer = area.to_crs(utm_data)


district_buffer["geometry"] = district_buffer.buffer(300)


district_buffer = district_buffer.to_crs(4326)

Загружаем данные по землепользованию.


In [ ]:
tags = {"landuse": True}

landuse = ox.features_from_polygon(
    district_buffer.geometry.iloc[0],
    tags
)

Смотрим структуру.


In [ ]:
landuse.head()

Отображаем на карте


In [ ]:
landuse.explore(tiles="cartodbpositron")

#### 0.2.3. Здания


Читаем данные


In [ ]:
tags_building = {"building": True}
building = ox.features_from_place(place_name, tags_building)

Смотрим на структуру


In [ ]:
building.head()

Отображаем на карте


In [ ]:
building.explore(tiles="cartodbpositron")

## 1. Создание базовой карты

Прежде чем добавлять на карту тематические слои — границу района, землепользование, буферы и здания — нужно создать **основу карты**.

На этом шаге мы:

1. определим, в какой точке карта должна открываться;
2. зададим начальный масштаб;
3. выберем фоновую подложку;
4. создадим объект карты Folium, в который позже будем добавлять слои.


### 1.1. Определяем центр карты

Чтобы карта сразу открывалась на нужной территории, нужно задать её центр. Для этого мы вычислим геометрический центр района.


In [ ]:
center = area.geometry.centroid.iloc[0]
center_lat = center.y
center_lon = center.x

print(center_lat, center_lon)

### 1.2. Создаём объект карты

Теперь создадим базовую карту Folium:

- `folium.Map()` создаёт интерактивную веб-карту;
- `location` задаёт центр карты;
- `zoom_start` задаёт начальный уровень приближения (зума);
- `tiles` выбирает фоновую подложку;
- `control_scale` добавляет масштабную линейку на карту.


In [ ]:
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles="cartodbpositron",
    control_scale=True
)

m

## 2. Добавление слоёв

Базовая карта уже создана, но пока она содержит только фоновую подложку.

Теперь мы будем постепенно добавлять на неё пространственные данные в виде отдельных слоёв.




### 2.1. Границы района

Начнём с самого базового слоя — границы района. Он нам пригодится, чтобы обозначить территорию исследования


#### 2.1.1. Настройка стиля

Параметр `style_function` вызывается отдельно для каждого объекта GeoJSON (`feature`). Объект `feature` содержит геометрию и атрибуты объекта в `feature["properties"]`.

Функция стилизации должна возвращать словарь с параметрами отображения объекта на карте:

- `fillColor` — цвет заливки полигона;
- `color` — цвет границы;
- `weight` — толщина линии;
- `fillOpacity` — прозрачность заливки;
- `opacity` — прозрачность контура;
- `dashArray` — делает линию пунктирной.


In [ ]:
area_style = lambda feature: {
    "fillColor": "#64748B",
    "color": "#334155",
    "weight": 2,
    "fillOpacity": 0.03,
    "opacity": 0.65,
    "dashArray": "6, 5",
}

#### 2.1.2. Добавление GeoJSON-слоя

Добавим слой на карту


In [ ]:
folium.GeoJson(
    area,
    name="Граница района",
    style_function=area_style,
).add_to(m)

m

### 2.2. Землепользование

Теперь перейдём к более сложному тематическому слою — землепользованию. В отличие от предыдущих примеров, здесь данные содержат несколько категорий объектов с разными типами использования территории. Поэтому перед настройкой отображения сначала посмотрим, какие категории присутствуют в наборе данных, а при необходимости объединим некоторые из них в более крупные группы.

#### 2.2.1. Анализ категорий

Для начала определим, какие типы землепользования представлены в данных.


In [ ]:
landuse["landuse"].value_counts()

#### 2.2.2. Группировка категорий

Чтобы упростить визуализацию и сделать карту более читаемой, объединим близкие по смыслу категории в несколько укрупнённых групп. Для этого создадим словарь соответствий между исходными типами землепользования и новыми группами.


In [ ]:
landuse_groups = {
    "residential": "residential",

    "commercial": "commercial",
    "retail": "commercial",

    "industrial": "industrial",
    "construction": "industrial",
    "garages": "industrial",
    "brownfield": "industrial",

    "grass": "green",
    "recreation_ground": "green",
    "flowerbed": "green",
    "farmland": "green",
    "greenfield": "green",
    "forest": "green",

    "religious": "special",
    "cemetery": "special",
    "military": "special",
    "education": "special",
    "allotments": "special",
    "reservoir": "special",
    "institutional": "special",
}

#### 2.2.3. Цвета для групп

Для каждой группы зададим отдельный цвет.


In [ ]:
landuse_colors = {
    "residential": "#fdd49e",
    "commercial": "#fc8d59",
    "industrial": "#d7301f",
    "green": "#78c679",
    "special": "#756bb1",
}

#### 2.2.4. Функция стилизации

Теперь создадим функцию, которая будет автоматически назначать стиль каждому объекту слоя.

Функция `landuse_style(feature)` вызывается отдельно для каждого объекта GeoJSON.
В аргумент `feature` передаётся текущий объект слоя вместе с его геометрией и атрибутами. Атрибуты объекта находятся в словаре `feature["properties"]`.

Внутри функции происходит несколько шагов:

1. Из атрибутов объекта извлекается значение поля `landuse`.
2. По словарю `landuse_groups` определяется, к какой укрупнённой группе относится данный тип землепользования.
3. Для найденной группы подбирается цвет из словаря `landuse_colors`.
4. Функция возвращает словарь со стилем отображения объекта на карте.


In [ ]:
def landuse_style(feature):

    landuse_type = feature["properties"].get("landuse")

    landuse_class = landuse_groups.get(landuse_type, "other")

    color = landuse_colors.get(landuse_class, "#d9d9d9")

    return {
        "fillColor": color,
        "color": "#ffffff",
        "weight": 0.4,
        "fillOpacity": 0.25,
        "opacity": 0.6,
    }

#### 2.2.5. Настройка всплывающих подсказок

Добавим всплывающие подсказки (`tooltip`), которые будут отображаться при наведении курсора на объект.

В подсказке будет выводиться поле `landuse`, содержащее исходный тип землепользования.
Параметр `aliases` задаёт подпись поля, а `localize=True` позволяет корректно форматировать значения.


In [ ]:
landuse_tooltip = folium.GeoJsonTooltip(
    fields=["landuse"],
    aliases=["Тип землепользования:"],
    localize=True
)

#### 2.2.6. Добавление слоя на карту

Теперь добавим слой землепользования на карту. Для отображения используется `folium.GeoJson`, которому передаются данные слоя, функция стилизации и всплывающие подсказки.

Параметр `style_function=landuse_style` означает, что для каждого объекта GeoJSON будет вызываться созданная ранее функция стилизации. Она автоматически определяет цвет и параметры отображения объекта на основе его атрибутов.


In [ ]:
folium.GeoJson(
    landuse,
    name="Землепользование",
    style_function=landuse_style,
    tooltip=landuse_tooltip
).add_to(m)

m

### 2.3. Зоны доступности зелени

На моей карте я сделаю буферы с зонами доступности до зеленных объектов, которые нашел в землепользовании

#### 2.3.1. Строем буферы зон доступности зелени

Возьмем 100 метровый буфер вокруг зеленых зон





Мы специально берем не центроиды, потому что зоны зелени могут быть достаточно большие 


In [ ]:
landuse_buffer = landuse
#landuse_buffer['geometry'] = landuse_buffer.geometry.centroid




Удаляем лишние поля, оставляя только необходимые. 


In [ ]:
landuse_buffer["osm_id"] = landuse_buffer.index.get_level_values("id")
landuse_buffer = landuse_buffer[["osm_id", "name", "geometry", "landuse"]]
landuse_buffer.head()

Перепроецирование данных

Для построения буферных зон данные должны быть в **проецированной системе координат** (единицы — метры).

Определение подходящей CRS


In [ ]:
utm_data = landuse_buffer.estimate_utm_crs()
print(utm_data)

#### 2.3.2 Перепроецирование




In [ ]:

#buildings_utm = buildings.to_crs(utm_data)
landuse_buffer = landuse_buffer.to_crs(utm_data)



In [ ]:
landuse_buffer.head()


#### 2.3.3 Построение зон доступности






- Выбираем всё, что попало в группу 'green' (использовали тот же словарь что и для группироки landuse)
- Буфер задаем 200 метров, как адекватное расстояние которое я готов пройти чтобы увидеть зелень.
- Объединяем все буферы в один
- для правильного отображение перепроицируем данные в географическую систему координат

In [ ]:
landuse_buffer['group'] = landuse_buffer['landuse'].map(landuse_groups)


green_points = landuse_buffer[landuse_buffer['group'] == 'green'].copy()

green_points['geometry'] = green_points.buffer(200)

green_union = green_points.dissolve(by='group').reset_index()

green_union = green_union.to_crs(4326)
green_union.explore(tiles="cartodbpositron")

#### 2.3.4. Функция стилизации




In [ ]:
def green_union_style(feature):

    

    return {
        "fillColor": "#2ca02c",
        "color": "#1a5f1a",
        "weight": 2,
        "fillOpacity": 0.4,
    }

#### 2.3.5. Настройка всплывающих подсказок

Добавим всплывающие подсказки (`tooltip`), которые будут отображаться при наведении курсора на объект.

В подсказке будет выводиться поле `landuse`, содержащее исходный тип землепользования.
Параметр `aliases` задаёт подпись поля, а `localize=True` позволяет корректно форматировать значения.


In [ ]:
green_union_tooltip = folium.GeoJsonTooltip(
    fields=['group'],          
    aliases=['Категория:'],     
    localize=True
)

#### 2.3.6. Добавление слоя на карту

Теперь добавим слой землепользования на карту. Для отображения используется `folium.GeoJson`, которому передаются данные слоя, функция стилизации и всплывающие подсказки.

Параметр `style_function=green_union_style` означает, что для каждого объекта GeoJSON будет вызываться созданная ранее функция стилизации. Она автоматически определяет цвет и параметры отображения объекта на основе его атрибутов.


In [ ]:
folium.GeoJson(
    green_union,
    name="Зеленый буфер",
    style_function=green_union_style,
    tooltip=green_union_tooltip
).add_to(m)

m

### 2.4. Здания

Для начала разобьем по группам здания которые входят в буфер доступности зелени, и здания которые не входят в него

#### 2.4.1. Стилизация зданий
Создадим столбец с показателями True и False по тому входят они в буфер доступности зелени или нет

In [ ]:
building['access_cat1'] = building.geometry.intersects(green_union.geometry.iloc[0])
building.head()

#### 2.4.2 Создадим словарь цветов для True False

In [ ]:
color_tips = {
    True: "#4daf4a",  
    False: "#e31a1c",  
}


#### 2.4.3 Отфильтруем датасет: оставляем только объекты с типом Polygon или MultiPolygon

In [ ]:
building_only_polygons = building[building.geometry.type.isin(['Polygon', 'MultiPolygon'])]

#### 2.4.4 Функция стилизации 

In [ ]:

def building_style(feature):
    
    access_val = feature['properties'].get('access_cat1')
    
    fill_color = color_tips.get(access_val, None)
    
    return {
        'fillColor': fill_color,
        'color': '#333333',     
        'weight': 1,            
        'fillOpacity': 0.7      
    }


#### 2.4.5 Добавление слоя на карту

In [ ]:

folium.GeoJson(
    building_only_polygons,
    name="Доступность зданий",
    style_function=building_style,
    tooltip=folium.GeoJsonTooltip(
        fields=['access_cat1', 'building'], 
        aliases=['Доступность:', 'Тип здания:']
    )
).add_to(m)



m

## 3. Управление картой

После добавления всех тематических слоёв настроим элементы управления картой.

### 3.1 Переключатель слоёв

Добавляем панель управления слоями. С её помощью пользователь может включать и выключать отдельные слои карты.


In [ ]:
folium.LayerControl(collapsed=False).add_to(m)

`LayerControl` позволяет включать и выключать слои. Параметр `collapsed=False` делает панель слоёв раскрытой по умолчанию.


### 3.2 Координаты курсора

Добавляем отображение координат курсора. Теперь при наведении мыши на карту пользователь будет видеть широту и долготу точки.


In [ ]:
from folium.plugins import MousePosition

MousePosition().add_to(m)

### 3.3 Полноэкранный режим

Параметры:

- `position="bottomright"` размещает кнопку в правом нижнем углу;
- `title` задаёт подпись при наведении на кнопку;
- `title_cancel` задаёт подпись для выхода из полноэкранного режима;
- `force_separate_button=True` делает кнопку отдельным элементом интерфейса.


In [ ]:
from folium.plugins import Fullscreen

Fullscreen(
    position="bottomright",
    title="Открыть на весь экран",
    title_cancel="Выйти из полноэкранного режима",
    force_separate_button=True,
).add_to(m)

### 3.4 Мини-карта

Мини-карта помогает понять, где находится текущий фрагмент карты относительно более крупной территории.


In [ ]:
from folium.plugins import MiniMap


MiniMap(tile_layer="cartodbpositron", toggle_display=True).add_to(m)

## 4. Просмотр и сохранение карты

На этом этапе карта полностью готова.

Мы добавили:

- тематические слои;
- стилизацию объектов;
- всплывающие подсказки;
- элементы управления картой.

Теперь можно посмотреть итоговый результат и сохранить карту как HTML-файл.


### 4.1 Итоговая карта

Отображаем готовую интерактивную карту.


In [ ]:
m

### 4.2 Сохранение карты

Сохраняем карту в HTML-файл. После этого карту можно открыть в браузере


In [ ]:
 m.save("index.html")

## Итог

В этом разделе мы создали интерактивную веб-карту с несколькими слоями.

Мы:

- загрузили и проверили данные;
- создали базовую карту;
- добавили тематические слои;
- настроили стили и подсказки;
- добавили элементы управления;
- сохранили карту в HTML.
